# Modelado de Fatiga con Redes Convolucionales Temporales (TCN)

Este notebook contiene la explicación teórica, la revisión de literatura científica, la arquitectura detallada y la implementación paso a paso de una red **TCN (Temporal Convolutional Network)** en PyTorch (empleando convoluciones causales dilatadas y bloques residuales) para predecir los niveles continuos de fatiga física y mental del dataset **FatigueSet**.

---

## 1. Fundamentos Teóricos y Literatura de Referencia

Las redes convolucionales temporales (TCN) representan una alternativa eficiente y robusta a las arquitecturas recurrentes (LSTM/GRU) para el modelado de series de tiempo. Sus características definitorias son:

1. **Procesamiento Causal:** No hay fuga de información del futuro hacia el pasado. En cada instante $t$, la predicción solo depende de los instantes $t' \le t$.
2. **Convoluciones Dilatadas:** Permiten expandir el campo receptivo de forma exponencial con la profundidad de la red, permitiendo capturar dependencias a largo plazo de forma computacionalmente eficiente.

### Convolución Dilatada y Causalidad

Dada una señal temporal unidimensional $x \in \mathbb{R}^{L}$ y un filtro/kernel $f \in \mathbb{R}^K$ de tamaño $K$, la convolución dilatada 1D con un factor de dilación $d$ se define como:
$$(x \star_d f)_t = \sum_{i=0}^{K-1} f_i \cdot x_{t - d \cdot i}$$

Donde:
- $d$ es el factor de dilación (paso de muestreo sobre la entrada).
- El término $t - d \cdot i$ garantiza que sólo se utilicen los estados pasados, manteniendo la causalidad.

Para implementar esto en PyTorch con convoluciones estándar, realizamos un **relleno (padding) asimétrico a la izquierda** del tensor de entrada:
$$\text{Padding Izquierdo} = (K - 1) \cdot d$$
Con un relleno de este tamaño, una convolución sin padding adicional (`padding=0`) y con longitud de filtro $K$ produce una salida con la misma longitud que la entrada original $L$.

### Bloque Residual y Campo Receptivo

Para evitar el desvanecimiento de gradientes en TCNs profundas, se utiliza una arquitectura de bloque residual:
$$\text{Output} = \text{ReLU}(x + \text{Dropout}(\text{WeightNorm}(\text{CausalConv1d}(\text{Dropout}(\text{WeightNorm}(\text{CausalConv1d}(x)))))))$$

Si el número de canales de entrada difiere de los de salida, el término $x$ se proyecta mediante una convolución 1D con kernel de tamaño 1.

El **Campo Receptivo ($R$)** de una TCN con $N$ bloques residuales (donde cada bloque contiene dos convoluciones), donde el bloque $i$ tiene una dilación $d_i = 2^{i-1}$, y un kernel de tamaño $K$, se calcula como:
$$R = 1 + \sum_{i=1}^{N} 2 \cdot (K - 1) \cdot d_i$$

Para $K=3$ y 5 bloques residuales con dilaciones $[1, 2, 4, 8, 16]$:
$$R = 1 + 2 \cdot (3 - 1) \cdot (1 + 2 + 4 + 8 + 16) = 1 + 4 \cdot 31 = 125$$
Lo que cubre casi en su totalidad una secuencia temporal de 128 timesteps.

### Ventajas de la TCN frente a RNN/LSTM
1. **Paralelismo:** A diferencia de las RNNs donde las predicciones dependen de los estados ocultos del paso anterior ($O(L)$ secuencial), las convoluciones de la TCN se pueden calcular en paralelo para toda la secuencia.
2. **Control del Campo Receptivo:** Modificando $K$ y las dilaciones, podemos ajustar con precisión cuántos timesteps en el pasado ve la red.
3. **Gradientes Estables:** Al ser una arquitectura convolucional directa con conexiones residuales, se evitan los problemas del desvanecimiento y la explosión del gradiente comunes en BPTT.

---

### Diagrama de Flujo del Campo Receptivo y Causalidad (Mermaid)

```mermaid
graph TD
    subgraph "Padding Causal (Lado Izquierdo)"
        pad["Relleno de Ceros: (K-1)*d a la izquierda"]
    end
    
    subgraph "Convolución Dilatada (d=1, 2, 4...)"
        conv1["Filtro Causal Dilatado 1"]
        wn1["Weight Normalization"]
        act1["ReLU + Dropout"]
    end

    subgraph "Bloque Residual TCN"
        conv2["Filtro Causal Dilatado 2"]
        wn2["Weight Normalization"]
        act2["ReLU + Dropout"]
        res["Conexión Residual (Proyección 1x1 si se requiere)"]
        add["Suma: Output + Residual"]
        act_out["Activación Final (ReLU)"]
    end

    pad --> conv1
    conv1 --> wn1
    wn1 --> act1
    act1 --> conv2
    conv2 --> wn2
    wn2 --> act2
    act2 --> add
    res --> add
    add --> act_out
```

---

### Citas Bibliográficas Científicas

* **Bai, S., Kolter, J. Z., & Koltun, V. (2018).** *An Empirical Evaluation of Generic Convolutional and Recurrent Networks for Sequence Modeling*. arXiv preprint arXiv:1803.01271. [Enlace al Paper](https://arxiv.org/abs/1803.01271)
* **Oord, A. van den, et al. (2016).** *WaveNet: A Generative Model for Raw Audio*. arXiv preprint arXiv:1609.03499. [Enlace al Paper](https://arxiv.org/abs/1609.03499)

In [2]:
# SETUP e IMPORTACIONES
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Añadir fatigueset-lib al sys.path
lib_path = str(Path.cwd().parent / "fatigueset-lib")
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

from fatigueset import FatigueSetPipeline
from fatigueset.models import CustomTCNRegressor, FatigueSequenceDataset
from fatigueset.models.rnn import _prepare_target_table, _merge_raw_streams, _build_sequences

print("[OK] Imports completados y path configurado.")
print(f"Versión de PyTorch: {torch.__version__}")
print(f"Dispositivo actual: {'cuda' if torch.cuda.is_available() else 'cpu'}")

[OK] Imports completados y path configurado.
Versión de PyTorch: 2.6.0+cu124
Dispositivo actual: cuda


## 2. Configuración del Pipeline y Construcción de Secuencias

Cargamos los datos fisiológicos del dataset `fatigueset` y alineamos los flujos de sensores de pecho (Chest) y muñeca (Wrist) construyendo tensores secuenciales de 128 timesteps con un paso de 32.

In [3]:
# Configuración del dataset y pipeline
dataset_path = str(Path.cwd().parent / "fatigueset")
pipeline = FatigueSetPipeline(dataset_path=dataset_path, umbral_nulos=5.0)

print("Cargando dataset...")
raw = pipeline.cargar_dataset(verbose=False)

print("Preparando targets del dataframe ML...")
df_ml = pipeline.construir_dataset_ml(raw)
df_targets = _prepare_target_table(df_ml)

print("Combinando streams fisiológicos crudos (Chest y Wrist)...")
df_raw = _merge_raw_streams(raw)

# Parámetros de ventanas de secuencia temporal
seq_len = 128
step = 32

print(f"Construyendo secuencias de tamaño={seq_len} y paso={step}...")
X_arr, y_arr, groups, feature_cols = _build_sequences(
    df_raw=df_raw,
    df_targets=df_targets,
    seq_len=seq_len,
    step=step
)

print(f"[OK] Dimensiones de tensores construidos:")
print(f"  - X: {X_arr.shape} (Número de ventanas x seq_len x features)")
print(f"  - y: {y_arr.shape} (Número de ventanas x 2 targets)")
print(f"  - Columnas de sensores: {len(feature_cols)}")

Cargando dataset...
Preparando targets del dataframe ML...
Combinando streams fisiológicos crudos (Chest y Wrist)...
Construyendo secuencias de tamaño=128 y paso=32...
[OK] Dimensiones de tensores construidos:
  - X: (1306, 128, 23) (Número de ventanas x seq_len x features)
  - y: (1306, 2) (Número de ventanas x 2 targets)
  - Columnas de sensores: 23


## 3. División de Datos por Participante (Group Split)

Para prevenir la fuga de información, seleccionamos al participante `'01'` exclusivamente para la validación cruzada y el resto para entrenamiento.

In [4]:
train_idx = np.where(groups != '01')[0]
val_idx = np.where(groups == '01')[0]

X_train, y_train = X_arr[train_idx], y_arr[train_idx]
X_val, y_val = X_arr[val_idx], y_arr[val_idx]

train_dataset = FatigueSequenceDataset(X_train, y_train)
val_dataset = FatigueSequenceDataset(X_val, y_val)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

Train samples: 1184
Validation samples: 122


## 4. Inicialización del Regresor TCN

Instanciamos nuestro regresor TCN personalizado. Definimos 5 niveles de bloques con dilaciones $[1, 2, 4, 8, 16]$ y un tamaño de canal de 64, lo que nos da un campo receptivo de 125 instantes temporales.

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

input_size = len(feature_cols)
num_channels = [64, 64, 64, 64, 64]
kernel_size = 3
dropout = 0.2

model = CustomTCNRegressor(
    input_size=input_size,
    num_channels=num_channels,
    kernel_size=kernel_size,
    dropout=dropout,
    output_size=2
).to(device)

print(model)

c:\Users\egull\OneDrive\Documentos\Proyectos\tfg\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


CustomTCNRegressor(
  (tcn): TemporalConvNet(
    (network): Sequential(
      (0): TemporalBlock(
        (conv1): Conv1d(23, 64, kernel_size=(3,), stride=(1,))
        (relu1): ReLU()
        (dropout1): Dropout(p=0.2, inplace=False)
        (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,))
        (relu2): ReLU()
        (dropout2): Dropout(p=0.2, inplace=False)
        (downsample): Conv1d(23, 64, kernel_size=(1,), stride=(1,))
        (relu_out): ReLU()
      )
      (1): TemporalBlock(
        (conv1): Conv1d(64, 64, kernel_size=(3,), stride=(1,), dilation=(2,))
        (relu1): ReLU()
        (dropout1): Dropout(p=0.2, inplace=False)
        (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), dilation=(2,))
        (relu2): ReLU()
        (dropout2): Dropout(p=0.2, inplace=False)
        (relu_out): ReLU()
      )
      (2): TemporalBlock(
        (conv1): Conv1d(64, 64, kernel_size=(3,), stride=(1,), dilation=(4,))
        (relu1): ReLU()
        (dropout1): Dropout(p=0

## 5. Entrenamiento de Validación (5 Épocas)

Entrenamos el modelo durante 5 épocas empleando un optimizador Adam, una tasa de aprendizaje de $10^{-3}$ y gradient clipping para mantener la estabilidad del entrenamiento.

In [6]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 5
print("Iniciando entrenamiento...")

for epoch in range(1, epochs + 1):
    # Modo entrenamiento
    model.train()
    total_train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        
        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Modo validación
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            loss = loss_fn(preds, yb)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader) if len(val_loader) > 0 else 0.0
    
    print(f"Epoch {epoch}/{epochs} - Train Loss (MSE): {avg_train_loss:.6f} - Val Loss (MSE): {avg_val_loss:.6f}")

print("[OK] Entrenamiento finalizado correctamente.")

Iniciando entrenamiento...
Epoch 1/5 - Train Loss (MSE): 569.187428 - Val Loss (MSE): 416.766479
Epoch 2/5 - Train Loss (MSE): 401.752483 - Val Loss (MSE): 176.773790
Epoch 3/5 - Train Loss (MSE): 383.786859 - Val Loss (MSE): 134.426330
Epoch 4/5 - Train Loss (MSE): 367.990245 - Val Loss (MSE): 223.018993
Epoch 5/5 - Train Loss (MSE): 360.004314 - Val Loss (MSE): 141.372407
[OK] Entrenamiento finalizado correctamente.


## 6. Serialización del Modelo

Guardamos los pesos del modelo en el directorio `/models/deep_learning/`.

In [7]:
output_dir = Path.cwd().parent / "models" / "deep_learning"
output_dir.mkdir(parents=True, exist_ok=True)

model_path = output_dir / "tcn_fatigue_notebook.pt"
torch.save(model.state_dict(), model_path)

print(f"[OK] Modelo guardado exitosamente en: {model_path}")

[OK] Modelo guardado exitosamente en: c:\Users\egull\OneDrive\Documentos\Proyectos\tfg\models\deep_learning\tcn_fatigue_notebook.pt
